# 03 TabPFN

使用 `model_ready_games.csv` 訓練 TabPFN，並比較兩組特徵：

| 設定 | 說明 |
|---|---|
| Feature Set A | 全部 33 個數值特徵 |
| Feature Set B | 11 個 `_diff` 差值特徵 |
| Train | 2018-2024 |
| Test | 2025 |
| Output | `tabpfn_predictions_all.csv`, `tabpfn_predictions_diff.csv`, `tabpfn_metrics.csv` |

這份 notebook 使用本地端 `tabpfn`，不使用 `tabpfn-client` 雲端 API。若 PyTorch 偵測到 CUDA，TabPFN 會優先使用 GPU。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tabpfn import TabPFNClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

PROJECT_ROOT = next(
    path for path in [Path('..').resolve(), Path('.').resolve()]
    if (path / 'data' / 'processed' / 'model_ready_games.csv').exists()
)
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'model_ready_games.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
METRICS_DIR = OUTPUT_DIR / 'metrics'
PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
for path in [METRICS_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
LABEL = 'home_win'

ID_COLS = {'game_id', 'season_id', 'year', 'phase', 'date', 'home_team', 'away_team'}
DIFF_COLS = [
    'win_rate_diff', 'runs_scored_diff', 'run_diff_10',
    'starter_ERA_diff', 'starter_WHIP_diff', 'starter_FIP_diff',
    'lineup_OPS_diff', 'lineup_OBP_diff', 'lineup_SLG_diff',
    'bullpen_ERA_diff', 'bullpen_WHIP_diff',
]

C:\Users\Eric\anaconda3\envs\tabpfn312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 驗證 GPU 與資料

In [2]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Selected device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA is not available. TabPFN will run on CPU.')


if not DATA_PATH.exists():
    raise FileNotFoundError(f'找不到資料檔：{DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH, dtype={'year': str})
df = df[df['year'] != '2026'].copy()

missing_diff_cols = [c for c in DIFF_COLS if c not in df.columns]
if missing_diff_cols:
    raise ValueError(f'DIFF_COLS 缺少欄位：{missing_diff_cols}')

feature_cols = [c for c in df.columns if c not in ID_COLS and c != LABEL]
train = df[df['year'].isin([str(y) for y in range(2018, 2025)])].copy()
test = df[df['year'] == '2025'].copy()

if train.empty or test.empty:
    raise ValueError(f'切分後資料不足：train={len(train)}, test={len(test)}')

y_train = train[LABEL].astype(int)
y_test = test[LABEL].astype(int)

print(f'Train rows: {len(train)} (2018-2024)')
print(f'Test rows : {len(test)} (2025)')
print(f'Feature Set A columns: {len(feature_cols)}')
print(f'Feature Set B columns: {len(DIFF_COLS)}')
print('Class balance train:', y_train.value_counts(normalize=True).round(3).to_dict())
print('Class balance test :', y_test.value_counts(normalize=True).round(3).to_dict())

PyTorch version: 2.11.0+cu128
CUDA available: True
Selected device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Train rows: 1947 (2018-2024)
Test rows : 358 (2025)
Feature Set A columns: 33
Feature Set B columns: 11
Class balance train: {1: 0.538, 0: 0.462}
Class balance test : {1: 0.545, 0: 0.455}


## 2. 前處理

In [3]:
imp_a = SimpleImputer(strategy='median')
X_train_a = imp_a.fit_transform(train[feature_cols])
X_test_a = imp_a.transform(test[feature_cols])

imp_b = SimpleImputer(strategy='median')
X_train_b = imp_b.fit_transform(train[DIFF_COLS])
X_test_b = imp_b.transform(test[DIFF_COLS])

print('Missing values after imputation:')
print('A train/test:', np.isnan(X_train_a).sum(), np.isnan(X_test_a).sum())
print('B train/test:', np.isnan(X_train_b).sum(), np.isnan(X_test_b).sum())

Missing values after imputation:
A train/test: 0 0
B train/test: 0 0


## 3. 訓練與評估函式

In [4]:
def train_eval_tabpfn(name, X_train, X_test):
    model = TabPFNClassifier(device=DEVICE)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        'Model': 'TabPFN',
        'Feature_Set': name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_prob), 4),
        'F1': round(f1_score(y_test, y_pred), 4),
        'Brier_Score': round(brier_score_loss(y_test, y_prob), 4),
    }

    print(f'=== {name} ===')
    for key, value in metrics.items():
        print(f'{key:<15}: {value}')
    print('\nClassification report:')
    print(classification_report(y_test, y_pred, target_names=['Away win/Home lose', 'Home win']))

    cm = pd.DataFrame(
        confusion_matrix(y_test, y_pred),
        index=['Actual 0', 'Actual 1'],
        columns=['Pred 0', 'Pred 1'],
    )
    display(cm)

    return model, y_pred, y_prob, metrics

## 4. Feature Set A：全部 33 個特徵

In [5]:
model_a, y_pred_a, y_prob_a, metrics_a = train_eval_tabpfn('All 33', X_train_a, X_test_a)

=== All 33 ===
Model          : TabPFN
Feature_Set    : All 33
Accuracy       : 0.6927
AUC            : 0.7494
F1             : 0.7208
Brier_Score    : 0.2059

Classification report:
                    precision    recall  f1-score   support

Away win/Home lose       0.67      0.65      0.66       163
          Home win       0.71      0.73      0.72       195

          accuracy                           0.69       358
         macro avg       0.69      0.69      0.69       358
      weighted avg       0.69      0.69      0.69       358



,Pred 0,Pred 1
Actual 0,106,57
Actual 1,53,142


## 5. Feature Set B：11 個 diff 特徵

In [6]:
model_b, y_pred_b, y_prob_b, metrics_b = train_eval_tabpfn('Diff 11', X_train_b, X_test_b)

=== Diff 11 ===
Model          : TabPFN
Feature_Set    : Diff 11
Accuracy       : 0.6872
AUC            : 0.7415
F1             : 0.7186
Brier_Score    : 0.2085

Classification report:
                    precision    recall  f1-score   support

Away win/Home lose       0.66      0.63      0.65       163
          Home win       0.70      0.73      0.72       195

          accuracy                           0.69       358
         macro avg       0.68      0.68      0.68       358
      weighted avg       0.69      0.69      0.69       358



,Pred 0,Pred 1
Actual 0,103,60
Actual 1,52,143


## 6. 比較與輸出 CSV

In [7]:
compare = pd.DataFrame([metrics_a, metrics_b])
display(compare)

pred_a = test[['game_id', 'date', 'home_team', 'away_team', LABEL]].copy()
pred_a['tabpfn_pred'] = y_pred_a
pred_a['tabpfn_prob'] = np.round(y_prob_a, 4)
pred_a['tabpfn_correct'] = (y_pred_a == y_test.values).astype(int)
pred_a.to_csv(PREDICTIONS_DIR / 'tabpfn_predictions_all.csv', index=False, encoding='utf-8-sig')

pred_b = test[['game_id', 'date', 'home_team', 'away_team', LABEL]].copy()
pred_b['tabpfn_pred'] = y_pred_b
pred_b['tabpfn_prob'] = np.round(y_prob_b, 4)
pred_b['tabpfn_correct'] = (y_pred_b == y_test.values).astype(int)
pred_b.to_csv(PREDICTIONS_DIR / 'tabpfn_predictions_diff.csv', index=False, encoding='utf-8-sig')

compare.to_csv(METRICS_DIR / 'tabpfn_metrics.csv', index=False, encoding='utf-8-sig')

print('Saved: tabpfn_predictions_all.csv')
print('Saved: tabpfn_predictions_diff.csv')
print('Saved: tabpfn_metrics.csv')

,Model,Feature_Set,Accuracy,AUC,F1,Brier_Score
0,TabPFN,All 33,0.6927,0.7494,0.7208,0.2059
1,TabPFN,Diff 11,0.6872,0.7415,0.7186,0.2085


Saved: tabpfn_predictions_all.csv
Saved: tabpfn_predictions_diff.csv
Saved: tabpfn_metrics.csv
